# Random Forest

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, recall_score
import pandas as pd
import numpy as np


def train_random_forest(X, y): 
    depths = [100, 500, 1000, 2000]
    min_splits = [2, 5, 10, 15, 20]
    criterions  = ['gini', 'entropy']

    filas = []
    for criterion in criterions: 
        for min_split in min_splits: 
            for depth in depths: 
                
                accs_train, accs_val, f1s, recalls = [], [], [], []       
                kf = KFold(n_splits=5)
                
                for train_index, val_index in kf.split(X):
                    xtrain, xval = X[train_index], X[val_index]
                    ytrain, yval = y[train_index], y[val_index]
                    
                    model = RandomForestClassifier(
                        criterion=criterion,
                        min_samples_split=min_split,
                        max_depth=depth, 
                        random_state=0) 

                    model.fit(xtrain, ytrain)

                    predictions_train = model.predict(xtrain)
                    predictions_val = model.predict(xval)
                    
                    accs_train.append(accuracy_score(ytrain, predictions_train))
                    accs_val.append(accuracy_score(yval, predictions_val))
                    f1s.append(f1_score(yval, predictions_val))
                    recalls.append(recall_score(yval, predictions_val))
                    
                filas.append({
                    'maxima profundidad': depth,
                    'criterio de split': criterion, 
                    'minimo de muestras para split': min_split,
                    'media acc train': np.mean(accs_train),
                    'media acc val': np.mean(accs_val),
                    'IC acc val': np.std(accs_val),
                    'media f1': np.mean(f1s),
                    'IC f1': np.std(f1s),
                    'media recall': np.mean(recalls),
                    'IC recal': np.std(recalls)
                })

    results = pd.DataFrame(filas)
    return results.sort_values(by='media f1', ascending=False)

In [ ]:
# Para datos sin filtrar

from libs.data_configuration import get_data_partitioned

X_train, X_test, y_train, y_test = get_data_partitioned() # por defecto 20% para validacion

train_random_forest(X_train, y_train).head(10)

,maxima profundidad,criterio de split,minimo de muestras para split,media acc train,media acc val,IC acc val,media f1,IC f1,media recall,IC recal
23,2000,entropy,2,1.000000,0.981875,0.002054,0.982078,0.001720,0.981922,0.000971
22,1000,entropy,2,1.000000,0.981875,0.002054,0.982078,0.001720,0.981922,0.000971
21,500,entropy,2,1.000000,0.981875,0.002054,0.982078,0.001720,0.981922,0.000971
20,100,entropy,2,1.000000,0.981875,0.002054,0.982078,0.001720,0.981922,0.000971
0,100,gini,2,1.000000,0.981250,0.001425,0.981471,0.001068,0.982657,0.001841
1,500,gini,2,1.000000,0.981250,0.001425,0.981471,0.001068,0.982657,0.001841
2,1000,gini,2,1.000000,0.981250,0.001425,0.981471,0.001068,0.982657,0.001841
3,2000,gini,2,1.000000,0.981250,0.001425,0.981471,0.001068,0.982657,0.001841
27,2000,entropy,5,0.998656,0.980250,0.000848,0.980455,0.000949,0.981666,0.002633
26,1000,entropy,5,0.998656,0.980250,0.000848,0.980455,0.000949,0.981666,0.002633


In [5]:
# para datos filtrados

from libs.data_configuration import get_filtered_data_partitioned

X_train, X_test, y_train, y_test = get_filtered_data_partitioned() # 20% de los datos para test por defecto 

train_random_forest(X_train, y_train).head(10) # mostrar los 10 mejores

KeyboardInterrupt: 

Los dos mejores modelos son los que tienen los siguientes parametros (maxima profundidad, criterio de split, minimo de muestras para split), (100, gini, 2) y (500, gini 2) sucesivamente 

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from libs.linear_reduction import train_models

xtrain, xtest, ytrain, ytest = get_filtered_data_partitioned() 

print(f"---Iniciamos con una cantidad de {xtrain.shape[1]} features---")

models = [
    RandomForestClassifier(criterion='gini', max_depth=100, min_samples_split=2),
    RandomForestClassifier(criterion='gini', max_depth=500, min_samples_split=2)
]

scaler = StandardScaler() 
xtrain, xtest = scaler.fit_transform(xtrain), scaler.transform(xtest)

pca = PCA(n_components=.95) # conservar un 95% de variabilidad de los datos 
xtrain, xtest = pca.fit_transform(xtrain), pca.transform(xtest)

print(f"---Terminamos con una cantidad de {xtrain.shape[1]} features---")

train_models(xtrain, xtest, ytrain, ytest, models, ['criterion', 'max_depth', 'min_samples_split'])

---Iniciamos con una cantidad de 41 features---
---Terminamos con una cantidad de 32 features---


,criterion,max_depth,min_samples_split,acc train,acc test,f1_score,recall
0,gini,100,2,1.0,0.9585,0.957631,0.955193
1,gini,500,2,1.0,0.9555,0.954615,0.953157


In [7]:
from libs.linear_reduction import validate_umap_reduction
from libs.data_configuration import get_filtered_data_partitioned

xtrain, xtest, ytrain, ytest = get_filtered_data_partitioned()

validate_umap_reduction(xtrain, ytrain, models[0])

/home/brayan/dev/ml/ModelosProyectoAula/.venv/lib/python3.13/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/home/brayan/dev/ml/ModelosProyectoAula/.venv/lib/python3.13/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces di

,n_components,n_neighbors,min_dist,mean_train_acc,mean_test_acc,ic_acc,mean_f1,ic_f1,mean_recall,ic_recall
7,5,10,0.1,0.999870,0.923067,0.001865,0.919788,0.004658,0.921232,0.007716
19,20,10,0.1,0.999968,0.921767,0.005644,0.918743,0.008558,0.925106,0.004534
18,20,10,0.0,0.999968,0.919558,0.005861,0.916515,0.007752,0.922711,0.005699
21,20,15,0.1,0.999903,0.918778,0.006852,0.915666,0.009133,0.921613,0.006363
13,10,10,0.1,0.999968,0.918648,0.003639,0.915480,0.003864,0.919014,0.005340
15,10,15,0.1,0.999968,0.918129,0.005590,0.914800,0.007754,0.918282,0.012854
23,20,30,0.1,0.999968,0.916959,0.008285,0.914005,0.010035,0.922558,0.008475
6,5,10,0.0,1.000000,0.916829,0.008126,0.913250,0.012037,0.917307,0.013826
9,5,15,0.1,0.999935,0.915140,0.003146,0.912294,0.006560,0.922431,0.008349
17,10,30,0.1,0.999968,0.915010,0.004565,0.911607,0.006332,0.915137,0.006043


In [ ]:
validate_umap_reduction(xtrain, ytrain, models[1])

/home/brayan/dev/ml/ModelosProyectoAula/.venv/lib/python3.13/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/home/brayan/dev/ml/ModelosProyectoAula/.venv/lib/python3.13/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces di